In [ ]:
import os, sys

import pandas as pd
import numpy as np

import geopandas as gpd


import plotly.express as px
import json

import matplotlib.pyplot as plt

import time

import glob

In [ ]:
dirname = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/NERC_TADS'
filename = os.path.join(dirname,'TADS_Transformer_2024_20250506.xlsx')

In [ ]:
NERC_regions_file = '/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/data/nerc-regions-geojson.geojson'




In [ ]:
# Read NERC regions geometry
nerc_regions_gdf = gpd.read_file(NERC_regions_file)

# Keep geometries in WGS84 for mapping baselayers/other overlays
if nerc_regions_gdf.crs is not None and str(nerc_regions_gdf.crs) != "EPSG:4326":
    nerc_regions_gdf = nerc_regions_gdf.to_crs("EPSG:4326")

# Identify the region-name column (robust to schema differences)
candidate_columns = [
    "NERC_REGION", "NERC_Region", "NERC", "region", "Region", "NAME", "name", "NAME_1"
]
region_col = next((col for col in candidate_columns if col in nerc_regions_gdf.columns), None)

if region_col is None:
    non_geom_cols = [c for c in nerc_regions_gdf.columns if c != "geometry"]
    object_like = [
        c for c in non_geom_cols
        if nerc_regions_gdf[c].dtype == "object" and nerc_regions_gdf[c].nunique(dropna=True) <= 20
    ]
    if len(object_like) == 0:
        raise ValueError(f"Could not infer region column. Available columns: {list(nerc_regions_gdf.columns)}")
    region_col = object_like[0]

# Designated colors for region categories (save this mapping for later overlays)
base_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79"
]

regions_sorted = sorted(nerc_regions_gdf[region_col].dropna().astype(str).unique())
nerc_region_colors = {region: base_palette[i % len(base_palette)] for i, region in enumerate(regions_sorted)}
nerc_regions_gdf["plot_color"] = nerc_regions_gdf[region_col].astype(str).map(nerc_region_colors)

# Quick look at what was loaded
print(f"Loaded {len(nerc_regions_gdf)} polygons")
print(f"Using region column: {region_col}")
print("Region-color mapping:")
print(nerc_region_colors)

# Plot for CONUS/North America context
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(14, 10))
nerc_regions_gdf.plot(
    ax=ax,
    color=nerc_regions_gdf["plot_color"],
    edgecolor="black",
    linewidth=0.5,
    alpha=0.85,
)

legend_handles = [Patch(facecolor=nerc_region_colors[r], edgecolor="black", label=r) for r in regions_sorted]
ax.legend(
    handles=legend_handles,
    title="NERC Regions",
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    borderaxespad=0.0,
)

ax.set_title("NERC Regions (Designated Colors)", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(-170, -50)  # North America framing
ax.set_ylim(15, 75)
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Load outage data from local Excel
outage_df = pd.read_excel(filename)

print(f"Rows: {len(outage_df):,}")
print("Columns:")
print(list(outage_df.columns))
outage_df.head(3)

In [ ]:
# Discover workbook sheets and find one with outage timing columns
xls = pd.ExcelFile(filename)
print("Sheet names:", xls.sheet_names)

sheet_columns = {}
for sheet in xls.sheet_names:
    cols = pd.read_excel(filename, sheet_name=sheet, nrows=0).columns.tolist()
    sheet_columns[sheet] = cols

candidate_sheets = [
    s for s, cols in sheet_columns.items()
    if ("OutageStartDT" in cols or "TimeZoneCode" in cols)
]

print("Candidate outage sheets:", candidate_sheets)
for s in candidate_sheets:
    print(f"\n{s} columns:")
    print(sheet_columns[s])

In [ ]:
# Analyze outage records (uses sheet with OperationalCauseCodeName)
outage_sheet = "TF_NonAuto_Outages"
outage_df = pd.read_excel(filename, sheet_name=outage_sheet).copy()

# Parse outage start datetime
outage_df["OutageStartDT"] = pd.to_datetime(outage_df["OutageStartDT"], errors="coerce")

# Parse mixed-format OutageDurationNbr into hours
from datetime import datetime, time, timedelta
import re

def duration_to_hours(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, timedelta):
        return value.total_seconds() / 3600.0

    if isinstance(value, time):
        return value.hour + value.minute / 60.0 + value.second / 3600.0

    if isinstance(value, datetime):
        excel_origin = datetime(1899, 12, 30)
        delta = value - excel_origin
        return delta.total_seconds() / 3600.0

    if isinstance(value, str):
        text = value.strip()
        if text == "":
            return np.nan

        # pattern like '0058:21' => 58 hours, 21 minutes
        m_hhhh_mm = re.fullmatch(r"(\d{1,4}):(\d{2})", text)
        if m_hhhh_mm:
            hours = int(m_hhhh_mm.group(1))
            minutes = int(m_hhhh_mm.group(2))
            return hours + minutes / 60.0

        # pattern like 'HH:MM:SS'
        m_hh_mm_ss = re.fullmatch(r"(\d{1,3}):(\d{2}):(\d{2})", text)
        if m_hh_mm_ss:
            hours = int(m_hh_mm_ss.group(1))
            minutes = int(m_hh_mm_ss.group(2))
            seconds = int(m_hh_mm_ss.group(3))
            return hours + minutes / 60.0 + seconds / 3600.0

        td = pd.to_timedelta(text, errors="coerce")
        if pd.notna(td):
            return td.total_seconds() / 3600.0

    return np.nan

outage_df["OutageDurationHours"] = outage_df["OutageDurationNbr"].apply(duration_to_hours)

# Map TimeZoneCode -> IANA timezone for UTC conversion
# Uses NERC/utility-style "Prevailing Time" codes and standard abbreviations observed in this workbook.
timezone_map = {
    "EPT": "America/New_York",      # Eastern Prevailing Time (DST-aware)
    "EST": "America/New_York",      # Eastern local clock label in this dataset
    "CPT": "America/Chicago",       # Central Prevailing Time (DST-aware)
    "CST": "America/Chicago",       # Central local clock label in this dataset
    "MPT": "America/Denver",        # Mountain Prevailing Time (DST-aware)
    "MST": "America/Denver",        # Mountain local clock label in this dataset
    "PPT": "America/Los_Angeles",   # Pacific Prevailing Time (DST-aware)
    "PST": "America/Los_Angeles",   # Pacific local clock label in this dataset
    "AST": "America/Halifax",       # Atlantic Standard/utility Atlantic region
    "APT": "America/Halifax",       # Atlantic Prevailing Time (assumed)
    "GMT": "UTC",
    "UTC": "UTC",
}

outage_df["TimeZoneCode"] = outage_df["TimeZoneCode"].astype(str).str.strip().str.upper()
outage_df["tz_name"] = outage_df["TimeZoneCode"].map(timezone_map)

unknown_tz = sorted(outage_df.loc[outage_df["tz_name"].isna(), "TimeZoneCode"].dropna().unique().tolist())
if unknown_tz:
    print("Unmapped TimeZoneCode values:", unknown_tz)
else:
    print("All TimeZoneCode values mapped.")

print("TimeZoneCode -> timezone mapping used in this run:")
print(outage_df[["TimeZoneCode", "tz_name"]].drop_duplicates().sort_values("TimeZoneCode").to_string(index=False))

# Convert local OutageStartDT to UTC by timezone group
outage_df["OutageStartDT_UTC"] = pd.Series(pd.NaT, index=outage_df.index, dtype="datetime64[ns, UTC]")
valid_time_rows = outage_df["OutageStartDT"].notna() & outage_df["tz_name"].notna()

for tz_name, idx in outage_df[valid_time_rows].groupby("tz_name").groups.items():
    localized = outage_df.loc[idx, "OutageStartDT"].dt.tz_localize(
        tz_name,
        ambiguous="NaT",
        nonexistent="shift_forward",
    )
    outage_df.loc[idx, "OutageStartDT_UTC"] = localized.dt.tz_convert("UTC")

# Build region-code color mapping from NERC map names (parenthetical code in NAME)
region_name_to_code = (
    nerc_regions_gdf[[region_col]]
    .drop_duplicates()
    .assign(RegionCode=lambda d: d[region_col].astype(str).str.extract(r"\(([^)]+)\)")[0].str.strip())
)
region_code_colors = {
    rc: nerc_region_colors[name]
    for name, rc in zip(region_name_to_code[region_col], region_name_to_code["RegionCode"])
    if pd.notna(rc)
}

# Keep analyzable rows
analysis_df = outage_df.dropna(subset=["RegionCode", "OutageDurationHours"]).copy()
analysis_df = analysis_df[analysis_df["OutageDurationHours"] >= 0]
analysis_df["RegionCode"] = analysis_df["RegionCode"].astype(str).str.strip().str.upper()

# 1) Duration statistics by RegionCode
duration_stats = (
    analysis_df.groupby("RegionCode")["OutageDurationHours"]
    .agg(
        n_events="count",
        mean_duration_hours="mean",
        median_duration_hours="median",
        p90_duration_hours=lambda s: s.quantile(0.90),
        max_duration_hours="max",
    )
    .sort_values("n_events", ascending=False)
)

# 2) OperationalCauseCodeName statistics by RegionCode
cause_counts = (
    analysis_df.groupby(["RegionCode", "OperationalCauseCodeName"])
    .size()
    .rename("count")
    .reset_index()
)
cause_counts["region_total"] = cause_counts.groupby("RegionCode")["count"].transform("sum")
cause_counts["probability"] = cause_counts["count"] / cause_counts["region_total"]

top_causes = (
    cause_counts.sort_values(["RegionCode", "count"], ascending=[True, False])
    .groupby("RegionCode")
    .head(5)
    .reset_index(drop=True)
)

print("Duration statistics by RegionCode:")
display(duration_stats)
print("Top OperationalCauseCodeName categories by RegionCode:")
display(top_causes)

# 3) Probability density of outage duration (PDF)
def pdf_points(values, log_bins=True, bins=30, min_positive=1e-6):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals) & (vals > 0)]
    if len(vals) == 0:
        return np.array([]), np.array([])

    if log_bins:
        vmin = max(vals.min(), min_positive)
        vmax = vals.max()
        if vmin == vmax:
            vmin = vmin * 0.5
            vmax = vmax * 1.5
        bin_edges = np.logspace(np.log10(vmin), np.log10(vmax), bins + 1)
        counts, edges = np.histogram(vals, bins=bin_edges, density=True)
        centers = np.sqrt(edges[:-1] * edges[1:])
    else:
        counts, edges = np.histogram(vals, bins=bins, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])

    mask = counts > 0
    return centers[mask], counts[mask]

fig, ax = plt.subplots(figsize=(12, 8))

# Total points in black
all_durations = analysis_df["OutageDurationHours"].dropna().to_numpy()
x_all, y_all = pdf_points(all_durations, log_bins=True, bins=35)
if len(x_all) > 0:
    ax.scatter(x_all, y_all, color="black", s=30, label="Total")

# Region points in designated colors
for region_code, sub in analysis_df.groupby("RegionCode"):
    vals = sub["OutageDurationHours"].dropna().to_numpy()
    x, y = pdf_points(vals, log_bins=True, bins=30)
    if len(x) == 0:
        continue
    color = region_code_colors.get(region_code, "#999999")
    ax.scatter(x, y, color=color, s=24, alpha=0.85, label=region_code)

ax.set_title("Outage Duration Probability Density (PDF): Total vs RegionCode")
ax.set_xlabel("Outage duration (hours)")
ax.set_ylabel("Probability density")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, which="both", linestyle="--", alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="RegionCode")
plt.tight_layout()
plt.show()

In [ ]:
# Debug why analysis_df is empty
check_df = pd.read_excel(filename, sheet_name="TF_NonAuto_Outages").copy()

print("Rows:", len(check_df))
print("Null counts:")
print(check_df[["RegionCode", "OutageDurationNbr", "OperationalCauseCodeName", "OutageStartDT", "TimeZoneCode"]].isna().sum())

print("\nRegionCode unique sample:")
print(check_df["RegionCode"].dropna().astype(str).str.strip().str.upper().value_counts().head(20))

print("\nOutageDurationNbr dtype:", check_df["OutageDurationNbr"].dtype)
print("OutageDurationNbr sample values:")
print(check_df["OutageDurationNbr"].dropna().head(20).tolist())

dur_num = pd.to_numeric(check_df["OutageDurationNbr"], errors="coerce")
print("\nNumeric-convertible durations:", dur_num.notna().sum(), "of", len(check_df))

print("\nTimeZoneCode unique:")
print(sorted(check_df["TimeZoneCode"].dropna().astype(str).str.strip().str.upper().unique().tolist()))

In [ ]:
# Interactive plots (Plotly): NERC regions map + outage duration PDF

# --- 1) Interactive NERC region map ---
interactive_map_df = nerc_regions_gdf[[region_col, "geometry"]].copy().reset_index(drop=True)
interactive_map_df["region_name"] = interactive_map_df[region_col].astype(str)
interactive_map_df["feature_id"] = interactive_map_df.index.astype(str)

interactive_geojson = json.loads(interactive_map_df.to_json())

fig_map = px.choropleth(
    interactive_map_df,
    geojson=interactive_geojson,
    locations="feature_id",
    featureidkey="properties.feature_id",
    color="region_name",
    color_discrete_map=nerc_region_colors,
    hover_name="region_name",
    projection="mercator",
    title="Interactive NERC Regions (Designated Colors)",
)
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(
    width=1100,
    height=700,
    margin={"r": 10, "t": 50, "l": 10, "b": 10},
    legend_title_text="NERC Regions",
)
fig_map.show()


# --- 2) Interactive outage duration probability density (PDF) ---
def pdf_points_df(values, label, log_bins=True, bins=30, min_positive=1e-6):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals) & (vals > 0)]
    if len(vals) == 0:
        return pd.DataFrame(columns=["duration_hours", "pdf", "series"])

    if log_bins:
        vmin = max(vals.min(), min_positive)
        vmax = vals.max()
        if vmin == vmax:
            vmin = vmin * 0.5
            vmax = vmax * 1.5
        bin_edges = np.logspace(np.log10(vmin), np.log10(vmax), bins + 1)
        counts, edges = np.histogram(vals, bins=bin_edges, density=True)
        centers = np.sqrt(edges[:-1] * edges[1:])
    else:
        counts, edges = np.histogram(vals, bins=bins, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])

    mask = counts > 0
    return pd.DataFrame({"duration_hours": centers[mask], "pdf": counts[mask], "series": label})

pdf_parts = []
all_vals = analysis_df["OutageDurationHours"].dropna().to_numpy()
if len(all_vals) > 0:
    pdf_parts.append(pdf_points_df(all_vals, "Total", log_bins=True, bins=35))

for region_code, sub in analysis_df.groupby("RegionCode"):
    vals = sub["OutageDurationHours"].dropna().to_numpy()
    if len(vals) < 2:
        continue
    pdf_parts.append(pdf_points_df(vals, region_code, log_bins=True, bins=30))

pdf_plot_df = pd.concat(pdf_parts, ignore_index=True)

scatter_color_map = {**region_code_colors, "Total": "#000000"}

fig_pdf = px.scatter(
    pdf_plot_df,
    x="duration_hours",
    y="pdf",
    color="series",
    color_discrete_map=scatter_color_map,
    title="Interactive Outage Duration Probability Density (PDF): Total vs RegionCode",
    labels={"duration_hours": "Outage duration (hours)", "pdf": "Probability density", "series": "RegionCode"},
)
fig_pdf.update_layout(width=1100, height=700)
fig_pdf.update_xaxes(type="log")
fig_pdf.update_yaxes(type="log")
fig_pdf.show()

In [ ]:
# Auto outages analysis (same pipeline as Non-Auto)
outage_sheet_auto = "TF_Auto_Outages"
auto_df = pd.read_excel(filename, sheet_name=outage_sheet_auto).copy()

# Ensure helpers exist (for standalone execution)
if "duration_to_hours" not in globals():
    from datetime import datetime, time, timedelta
    import re

    def duration_to_hours(value):
        if pd.isna(value):
            return np.nan

        if isinstance(value, timedelta):
            return value.total_seconds() / 3600.0

        if isinstance(value, time):
            return value.hour + value.minute / 60.0 + value.second / 3600.0

        if isinstance(value, datetime):
            excel_origin = datetime(1899, 12, 30)
            delta = value - excel_origin
            return delta.total_seconds() / 3600.0

        if isinstance(value, str):
            text = value.strip()
            if text == "":
                return np.nan

            m_hhhh_mm = re.fullmatch(r"(\d{1,4}):(\d{2})", text)
            if m_hhhh_mm:
                hours = int(m_hhhh_mm.group(1))
                minutes = int(m_hhhh_mm.group(2))
                return hours + minutes / 60.0

            m_hh_mm_ss = re.fullmatch(r"(\d{1,3}):(\d{2}):(\d{2})", text)
            if m_hh_mm_ss:
                hours = int(m_hh_mm_ss.group(1))
                minutes = int(m_hh_mm_ss.group(2))
                seconds = int(m_hh_mm_ss.group(3))
                return hours + minutes / 60.0 + seconds / 3600.0

            td = pd.to_timedelta(text, errors="coerce")
            if pd.notna(td):
                return td.total_seconds() / 3600.0

        return np.nan

if "timezone_map" not in globals():
    timezone_map = {
        "EPT": "America/New_York",
        "EST": "America/New_York",
        "CPT": "America/Chicago",
        "CST": "America/Chicago",
        "MPT": "America/Denver",
        "MST": "America/Denver",
        "PPT": "America/Los_Angeles",
        "PST": "America/Los_Angeles",
        "AST": "America/Halifax",
        "APT": "America/Halifax",
        "GMT": "UTC",
        "UTC": "UTC",
    }

if "pdf_points" not in globals():
    def pdf_points(values, log_bins=True, bins=30, min_positive=1e-6):
        vals = np.asarray(values, dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals) == 0:
            return np.array([]), np.array([])

        if log_bins:
            vmin = max(vals.min(), min_positive)
            vmax = vals.max()
            if vmin == vmax:
                vmin = vmin * 0.5
                vmax = vmax * 1.5
            bin_edges = np.logspace(np.log10(vmin), np.log10(vmax), bins + 1)
            counts, edges = np.histogram(vals, bins=bin_edges, density=True)
            centers = np.sqrt(edges[:-1] * edges[1:])
        else:
            counts, edges = np.histogram(vals, bins=bins, density=True)
            centers = 0.5 * (edges[:-1] + edges[1:])

        mask = counts > 0
        return centers[mask], counts[mask]

# Parse outage start datetime and duration
auto_df["OutageStartDT"] = pd.to_datetime(auto_df["OutageStartDT"], errors="coerce")
auto_df["OutageDurationHours"] = auto_df["OutageDurationNbr"].apply(duration_to_hours)

# Timezone conversion to UTC
auto_df["TimeZoneCode"] = auto_df["TimeZoneCode"].astype(str).str.strip().str.upper()
auto_df["tz_name"] = auto_df["TimeZoneCode"].map(timezone_map)

unknown_tz_auto = sorted(auto_df.loc[auto_df["tz_name"].isna(), "TimeZoneCode"].dropna().unique().tolist())
if unknown_tz_auto:
    print("Unmapped TimeZoneCode values (auto):", unknown_tz_auto)
else:
    print("All TimeZoneCode values mapped for auto outages.")

auto_df["OutageStartDT_UTC"] = pd.Series(pd.NaT, index=auto_df.index, dtype="datetime64[ns, UTC]")
valid_time_rows_auto = auto_df["OutageStartDT"].notna() & auto_df["tz_name"].notna()

for tz_name, idx in auto_df[valid_time_rows_auto].groupby("tz_name").groups.items():
    localized = auto_df.loc[idx, "OutageStartDT"].dt.tz_localize(
        tz_name,
        ambiguous="NaT",
        nonexistent="shift_forward",
    )
    auto_df.loc[idx, "OutageStartDT_UTC"] = localized.dt.tz_convert("UTC")

# Keep analyzable rows
auto_analysis_df = auto_df.dropna(subset=["RegionCode", "OutageDurationHours"]).copy()
auto_analysis_df = auto_analysis_df[auto_analysis_df["OutageDurationHours"] >= 0]
auto_analysis_df["RegionCode"] = auto_analysis_df["RegionCode"].astype(str).str.strip().str.upper()

# 1) Duration statistics by RegionCode (Auto outages)
auto_duration_stats = (
    auto_analysis_df.groupby("RegionCode")["OutageDurationHours"]
    .agg(
        n_events="count",
        mean_duration_hours="mean",
        median_duration_hours="median",
        p90_duration_hours=lambda s: s.quantile(0.90),
        max_duration_hours="max",
    )
    .sort_values("n_events", ascending=False)
)

print("Auto outages: duration statistics by RegionCode")
display(auto_duration_stats)

# 2) Cause statistics by RegionCode (Auto outages)
auto_cause_fields = ["InitiationCauseCodeName", "SustainedCauseCodeName"]

for cause_field in auto_cause_fields:
    if cause_field not in auto_analysis_df.columns:
        continue

    auto_cause_counts = (
        auto_analysis_df.groupby(["RegionCode", cause_field])
        .size()
        .rename("count")
        .reset_index()
    )
    auto_cause_counts["region_total"] = auto_cause_counts.groupby("RegionCode")["count"].transform("sum")
    auto_cause_counts["probability"] = auto_cause_counts["count"] / auto_cause_counts["region_total"]

    auto_top_causes = (
        auto_cause_counts.sort_values(["RegionCode", "count"], ascending=[True, False])
        .groupby("RegionCode")
        .head(5)
        .reset_index(drop=True)
    )

    print(f"\nAuto outages: top {cause_field} categories by RegionCode")
    display(auto_top_causes)

# 3) Probability density of outage duration (PDF) for auto outages
fig, ax = plt.subplots(figsize=(12, 8))

all_auto = auto_analysis_df["OutageDurationHours"].dropna().to_numpy()
x_all, y_all = pdf_points(all_auto, log_bins=True, bins=35)
if len(x_all) > 0:
    ax.scatter(x_all, y_all, color="black", s=30, label="Total")

for region_code, sub in auto_analysis_df.groupby("RegionCode"):
    vals = sub["OutageDurationHours"].dropna().to_numpy()
    x, y = pdf_points(vals, log_bins=True, bins=30)
    if len(x) == 0:
        continue
    color = region_code_colors.get(region_code, "#999999")
    ax.scatter(x, y, color=color, s=24, alpha=0.85, label=region_code)

ax.set_title("Auto Outages: Duration Probability Density (PDF)")
ax.set_xlabel("Outage duration (hours)")
ax.set_ylabel("Probability density")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, which="both", linestyle="--", alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="RegionCode")
plt.tight_layout()
plt.show()

In [ ]:
# Disaggregation by alias-based entity/element identifiers

# Load sheets if not already in memory
if "outage_df" not in globals() or outage_df is None:
    outage_df = pd.read_excel(filename, sheet_name="TF_NonAuto_Outages").copy()
if "auto_df" not in globals() or auto_df is None:
    auto_df = pd.read_excel(filename, sheet_name="TF_Auto_Outages").copy()

# Build a combined table for alias-based summary
nonauto_tagged = outage_df.copy()
nonauto_tagged["sheet"] = "TF_NonAuto_Outages"
auto_tagged = auto_df.copy()
auto_tagged["sheet"] = "TF_Auto_Outages"

combined = pd.concat([nonauto_tagged, auto_tagged], ignore_index=True)

# Canonical entity identifier: prefer ElementIdentifierName_AliasID, fallback to AliasID
combined["entity_id"] = (
    combined.get("ElementIdentifierName_AliasID")
    .where(combined.get("ElementIdentifierName_AliasID").notna(), combined.get("AliasID"))
    .astype(str)
)
combined["entity_id"] = combined["entity_id"].replace("nan", np.nan)

# Basic disaggregation coverage
print("Unique entity_id counts by sheet:")
display(combined.groupby("sheet")["entity_id"].nunique(dropna=True).rename("n_entities"))

# Entity counts by region
if "RegionCode" in combined.columns:
    entity_by_region = (
        combined.dropna(subset=["RegionCode", "entity_id"])
        .groupby(["sheet", "RegionCode"])["entity_id"]
        .nunique()
        .rename("n_entities")
        .reset_index()
        .sort_values(["sheet", "n_entities"], ascending=[True, False])
    )
    print("\nEntity disaggregation by RegionCode:")
    display(entity_by_region)

# Entity counts by voltage class
if "VoltageClassCodeName" in combined.columns:
    entity_by_voltage = (
        combined.dropna(subset=["VoltageClassCodeName", "entity_id"])
        .groupby(["sheet", "VoltageClassCodeName"])["entity_id"]
        .nunique()
        .rename("n_entities")
        .reset_index()
        .sort_values(["sheet", "n_entities"], ascending=[True, False])
    )
    print("\nEntity disaggregation by VoltageClassCodeName:")
    display(entity_by_voltage)

# Overlap of alias IDs across sheets (shared entities)
nonauto_entities = set(nonauto_tagged.get("ElementIdentifierName_AliasID", nonauto_tagged.get("AliasID")).dropna().astype(str))
auto_entities = set(auto_tagged.get("ElementIdentifierName_AliasID", auto_tagged.get("AliasID")).dropna().astype(str))
shared_entities = nonauto_entities.intersection(auto_entities)

print("\nEntity overlap between sheets:")
print({
    "nonauto_entities": len(nonauto_entities),
    "auto_entities": len(auto_entities),
    "shared_entities": len(shared_entities),
})

In [ ]:
# Asset-level behavior summary

# Ensure inputs are available
if "outage_df" not in globals() or outage_df is None:
    outage_df = pd.read_excel(filename, sheet_name="TF_NonAuto_Outages").copy()
if "auto_df" not in globals() or auto_df is None:
    auto_df = pd.read_excel(filename, sheet_name="TF_Auto_Outages").copy()

# Ensure helper for duration parsing exists
if "duration_to_hours" not in globals():
    from datetime import datetime, time, timedelta
    import re

    def duration_to_hours(value):
        if pd.isna(value):
            return np.nan

        if isinstance(value, timedelta):
            return value.total_seconds() / 3600.0

        if isinstance(value, time):
            return value.hour + value.minute / 60.0 + value.second / 3600.0

        if isinstance(value, datetime):
            excel_origin = datetime(1899, 12, 30)
            delta = value - excel_origin
            return delta.total_seconds() / 3600.0

        if isinstance(value, str):
            text = value.strip()
            if text == "":
                return np.nan

            m_hhhh_mm = re.fullmatch(r"(\d{1,4}):(\d{2})", text)
            if m_hhhh_mm:
                hours = int(m_hhhh_mm.group(1))
                minutes = int(m_hhhh_mm.group(2))
                return hours + minutes / 60.0

            m_hh_mm_ss = re.fullmatch(r"(\d{1,3}):(\d{2}):(\d{2})", text)
            if m_hh_mm_ss:
                hours = int(m_hh_mm_ss.group(1))
                minutes = int(m_hh_mm_ss.group(2))
                seconds = int(m_hh_mm_ss.group(3))
                return hours + minutes / 60.0 + seconds / 3600.0

            td = pd.to_timedelta(text, errors="coerce")
            if pd.notna(td):
                return td.total_seconds() / 3600.0

        return np.nan

# Build a combined event table with canonical entity_id
nonauto_events = outage_df.copy()
nonauto_events["sheet"] = "TF_NonAuto_Outages"
auto_events = auto_df.copy()
auto_events["sheet"] = "TF_Auto_Outages"

combined_events = pd.concat([nonauto_events, auto_events], ignore_index=True)

combined_events["entity_id"] = (
    combined_events.get("ElementIdentifierName_AliasID")
    .where(combined_events.get("ElementIdentifierName_AliasID").notna(), combined_events.get("AliasID"))
    .astype(str)
)
combined_events["entity_id"] = combined_events["entity_id"].replace("nan", np.nan)

combined_events["OutageDurationHours"] = combined_events["OutageDurationNbr"].apply(duration_to_hours)

# Keep analyzable asset-level records
asset_events = combined_events.dropna(subset=["entity_id", "OutageDurationHours"]).copy()
asset_events = asset_events[asset_events["OutageDurationHours"] >= 0]

# Asset-level summary statistics
asset_stats = (
    asset_events.groupby(["entity_id", "sheet"])["OutageDurationHours"]
    .agg(
        n_outages="count",
        total_duration_hours="sum",
        mean_duration_hours="mean",
        median_duration_hours="median",
        p90_duration_hours=lambda s: s.quantile(0.90),
        max_duration_hours="max",
    )
    .reset_index()
)

print("Asset-level summary (per entity_id and sheet):")
display(asset_stats.head(10))

# Top assets by total duration
print("\nTop assets by total duration (hours):")
display(asset_stats.sort_values("total_duration_hours", ascending=False).head(15))

# Top assets by outage count
print("\nTop assets by outage count:")
display(asset_stats.sort_values("n_outages", ascending=False).head(15))

# Scatter: outage count vs total duration (log-log)
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    asset_stats["n_outages"],
    asset_stats["total_duration_hours"],
    s=20,
    alpha=0.6,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Outage count per asset")
ax.set_ylabel("Total outage duration (hours)")
ax.set_title("Asset-Level Behavior: Count vs Total Duration")
ax.grid(True, which="both", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

# Optional: by region if available
if "RegionCode" in asset_events.columns:
    asset_by_region = (
        asset_events.groupby(["RegionCode", "entity_id"])["OutageDurationHours"]
        .agg(n_outages="count", total_duration_hours="sum")
        .reset_index()
    )
    print("\nAsset-level count by RegionCode (top 10 by duration):")
    display(asset_by_region.sort_values("total_duration_hours", ascending=False).head(10))